In [1]:
import torch
print(torch.cuda.is_available())

True


===============================
모델 저장 및 예측 결과 나란히 저장

In [ ]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
from PIL import Image
import numpy as np
import imageio
import matplotlib.pyplot as plt
import segmentation_models_pytorch as smp
from tqdm import tqdm
import random

# RGB 이미지와 희소(depth_gt_sparse) 뎁스맵을 불러오는 커스텀 데이터셋 정의
class DepthDataset(Dataset):
    def __init__(self, indices, rgb_dir='output/rgb', depth_dir='output/depth_gt_sparse', img_size=(480, 640)):
        self.indices = indices
        self.rgb_dir = rgb_dir
        self.depth_dir = depth_dir
        self.img_size = img_size
        self.transform = transforms.Compose([
            transforms.Resize(img_size),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])  # ImageNet 정규화
        ])

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        index = self.indices[idx]
        rgb_path = os.path.join(self.rgb_dir, f'{index:06d}.png')
        depth_path = os.path.join(self.depth_dir, f'{index:06d}.png')

        rgb = Image.open(rgb_path).convert('RGB')
        rgb = self.transform(rgb)

        depth = imageio.imread(depth_path).astype(np.float32) / 1000.0
        depth = torch.from_numpy(depth).unsqueeze(0).unsqueeze(0)
        depth = F.interpolate(depth, size=self.img_size, mode='nearest').squeeze(0)
        mask = (depth > 0).float()

        return rgb, depth, mask

class DepthEstimationModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.unet = smp.Unet(
            encoder_name='resnet18',
            encoder_weights='imagenet',
            in_channels=3,
            classes=1
        )
        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.unet(x)
        x = self.relu(x)
        return x

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

model = DepthEstimationModel().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

# train_indices = list(range(1000))
# val_indices = list(range(1000, 1200))

random.seed(42)
# 전체 데이터 수 (예: output/rgb 내 이미지 개수 기준)
total_data = len(os.listdir('output/rgb'))  # 예: 1200

# 전체 인덱스를 무작위로 섞음
all_indices = list(range(total_data))
random.shuffle(all_indices)

# 학습 80%, 검증 20%
split_ratio = 0.8
split_idx = int(total_data * split_ratio)

train_indices = all_indices[:split_idx]
val_indices   = all_indices[split_idx:]

train_dataset = DepthDataset(train_indices)
val_dataset = DepthDataset(val_indices)
train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=8, shuffle=False)

num_epochs = 20
for epoch in range(num_epochs):
    model.train()
    train_loss = 0
    print(f"Epoch {epoch+1}/{num_epochs} 시작")
    for rgb, depth, mask in tqdm(train_loader, desc=f"Train Epoch {epoch+1}"):
        rgb, depth, mask = rgb.to(device), depth.to(device), mask.to(device)

        optimizer.zero_grad()
        pred = model(rgb)
        loss = ((pred - depth).abs() * mask).sum() / mask.sum()
        loss.backward()
        optimizer.step()
        train_loss += loss.item()

    train_loss /= len(train_loader)

    model.eval()
    val_loss = 0
    val_rmse = 0
    val_logrmse = 0
    val_delta1 = 0
    val_delta2 = 0
    val_delta3 = 0
    total_pixels = 0

    with torch.no_grad():
        for rgb, depth, mask in val_loader:
            rgb, depth, mask = rgb.to(device), depth.to(device), mask.to(device)
            pred = model(rgb)

            abs_error = (pred - depth).abs()
            squared_error = ((pred - depth) ** 2) * mask
            rmse = torch.sqrt(squared_error.sum() / mask.sum())
            log_error = (torch.log(pred + 1e-6) - torch.log(depth + 1e-6)) ** 2
            logrmse = torch.sqrt((log_error * mask).sum() / mask.sum())

            ratio = torch.max(pred / (depth + 1e-8), depth / (pred + 1e-8))
            delta1 = (ratio < 1.25).float() * mask
            delta2 = (ratio < 1.25 ** 2).float() * mask
            delta3 = (ratio < 1.25 ** 3).float() * mask

            val_delta1 += delta1.sum().item()
            val_delta2 += delta2.sum().item()
            val_delta3 += delta3.sum().item()
            total_pixels += mask.sum().item()

            val_loss += (abs_error * mask).sum().item() / mask.sum().item()
            val_rmse += rmse.item()
            val_logrmse += logrmse.item()

    val_loss /= len(val_loader)
    val_rmse /= len(val_loader)
    val_logrmse /= len(val_loader)
    val_delta1 /= total_pixels
    val_delta2 /= total_pixels
    val_delta3 /= total_pixels

    print(f"Epoch {epoch+1}/{num_epochs}, Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}, RMSE: {val_rmse:.4f}, logRMSE: {val_logrmse:.4f}, δ1: {val_delta1:.4f}, δ2: {val_delta2:.4f}, δ3: {val_delta3:.4f}\n")

# 학습된 모델 저장
os.makedirs('output/model', exist_ok=True)
torch.save(model.state_dict(), 'output/model/depth_model.pth')

# 예측 결과 저장 (RGB + 예측 뎁스 시각화)
os.makedirs('output/pred_depth', exist_ok=True)
model.eval()
with torch.no_grad():
    for i in range(10):
        rgb_tensor, _, _ = val_dataset[i]
        rgb = rgb_tensor.unsqueeze(0).to(device)
        pred = model(rgb)
        pred_np = pred.squeeze().cpu().numpy()

        # RGB 이미지 복원
        inv_transform = transforms.Normalize(
            mean=[-m/s for m, s in zip([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])],
            std=[1/s for s in [0.229, 0.224, 0.225]]
        )
        rgb_denorm = inv_transform(rgb_tensor).clamp(0, 1).permute(1, 2, 0).cpu().numpy()

        # 예측된 깊이 맵 컬러맵 적용
        pred_color = plt.cm.viridis(pred_np / pred_np.max())[:, :, :3]  # RGB만
        vis = np.hstack((rgb_denorm, pred_color))
        vis = (vis * 255).astype(np.uint8)

        imageio.imwrite(f'output/pred_depth/{val_indices[i]:06d}_vis.png', vis)
        pred_mm = (pred_np * 1000.0).clip(0, 65535).astype(np.uint16)
        imageio.imwrite(f'output/pred_depth/{val_indices[i]:06d}.png', pred_mm)

print("Training completed. Model saved to 'output/model/depth_model.pth' and depth maps to 'output/pred_depth'.")


Epoch 1/20 시작


Train Epoch 1:   0%|          | 0/1265 [00:00<?, ?it/s]C:\Users\ADMIN\AppData\Local\Temp\ipykernel_19452\1899088319.py:39: DeprecationWarning: Starting with ImageIO v3 the behavior of this function will switch to that of iio.v3.imread. To keep the current behavior (and make this warning disappear) use `import imageio.v2 as imageio` or call `imageio.v2.imread` directly.
  depth = imageio.imread(depth_path).astype(np.float32) / 1000.0
Train Epoch 1:   2%|▏         | 20/1265 [00:16<17:06,  1.21it/s]

In [6]:
import matplotlib.pyplot as plt
import cv2

# 예측 결과 저장 디렉터리
os.makedirs('output/pred_vis', exist_ok=True)

model.eval()
with torch.no_grad():
    for i in range(10):
        rgb_tensor, _, _ = val_dataset[i]
        rgb_input = rgb_tensor.unsqueeze(0).to(device)
        pred = model(rgb_input).squeeze().cpu().numpy()

        # 정규화
        pred_norm = (pred - pred.min()) / (pred.max() - pred.min() + 1e-6)  # [0, 1] 정규화
        pred_uint8 = (pred_norm * 255).astype(np.uint8)                    # [0, 255]로 변환

        # 컬러맵 적용: JET (빨강-노랑-파랑 계열)
        pred_color = cv2.applyColorMap(pred_uint8, cv2.COLORMAP_JET)
        pred_color = cv2.cvtColor(pred_color, cv2.COLOR_BGR2RGB)          # OpenCV는 BGR -> RGB 변환

        # 원본 RGB 되돌리기 (정규화 해제)
        rgb_np = rgb_tensor.permute(1, 2, 0).cpu().numpy()
        rgb_np = (rgb_np * [0.229, 0.224, 0.225]) + [0.485, 0.456, 0.406]  # ImageNet unnormalize
        rgb_np = (rgb_np * 255).clip(0, 255).astype(np.uint8)

        # 좌우로 붙이기
        concat = np.concatenate([rgb_np, pred_color], axis=1)
        Image.fromarray(concat).save(f'output/pred_vis/{val_indices[i]:06d}.png')


C:\Users\ADMIN\AppData\Local\Temp\ipykernel_19452\2799130129.py:38: DeprecationWarning: Starting with ImageIO v3 the behavior of this function will switch to that of iio.v3.imread. To keep the current behavior (and make this warning disappear) use `import imageio.v2 as imageio` or call `imageio.v2.imread` directly.
  depth = imageio.imread(depth_path).astype(np.float32) / 1000.0
